# 001 — Byte-Pair Encoding From Scratch

This notebook builds a real Byte-Pair Encoding (BPE) tokenizer in plain Python — no
`tokenizers`/HuggingFace library — trains it on a small toy corpus, inspects the merges
it actually learns, tokenizes a new sentence, and verifies encode-then-decode is a true
round trip. See `notes.md` for the full derivation; this notebook is the from-scratch
implementation and the experiment referenced there.

In [1]:
import re
import collections

WORD_END = "</w>"  # explicit end-of-word marker, so BPE never merges across a word boundary


## 1. The toy training corpus

Deliberately small and repetitive (a few hundred words) so merges are easy to trace by
hand and the whole training loop runs in well under a second — this is a toy proving the
mechanism, not a production tokenizer (see `notes.md`, "Practical implementation", for
the contrast with a tokenizer trained on billions of tokens).

In [2]:
CORPUS = """
the quick brown fox jumps over the lazy dog
the quick brown fox runs past the lazy dog
a lazy dog sleeps while the quick fox watches
the tokenizer splits the lower into low and er
the tokenizer splits the lowest into low and est
lower and lowest and lower again
newer and newest words appear in the newer texts
the model tokenizes the text using a learned tokenizer
tokenization turns text into tokens for the model
the quick fox and the lazy dog are old friends
""".strip()

print(f"Corpus length: {len(CORPUS)} characters, {len(CORPUS.split())} words (whitespace-split)")


Corpus length: 462 characters, 87 words (whitespace-split)


## 2. Step 1 of the algorithm: represent every word as a sequence of characters

Every word is split into its individual characters, plus an explicit end-of-word symbol
`</w>`. The end-of-word symbol matters: without it, BPE could merge the last character of
one word with the first character of the next (they're adjacent once whitespace is
dropped), which would make tokens leak across word boundaries. Word frequencies are
counted once up front, since BPE always operates on *counts* of symbol pairs, never on
raw un-weighted text.

In [3]:
def word_freqs(corpus: str) -> dict[tuple[str, ...], int]:
    """Pre-tokenize on whitespace; represent each word as a tuple of characters with
    an explicit end-of-word marker; count how often each such tuple occurs."""
    freqs: dict[tuple[str, ...], int] = collections.Counter()
    for word in re.findall(r"\S+", corpus.lower()):
        symbols = tuple(list(word) + [WORD_END])
        freqs[symbols] += 1
    return freqs


freqs = word_freqs(CORPUS)
print(f"{len(freqs)} distinct words")
for symbols, count in sorted(freqs.items(), key=lambda kv: -kv[1])[:6]:
    print(count, symbols)


42 distinct words
15 ('t', 'h', 'e', '</w>')
6 ('a', 'n', 'd', '</w>')
4 ('q', 'u', 'i', 'c', 'k', '</w>')
4 ('f', 'o', 'x', '</w>')
4 ('l', 'a', 'z', 'y', '</w>')
4 ('d', 'o', 'g', '</w>')


## 3. Step 2: count adjacent symbol pairs, merge the most frequent one

This is the entire algorithm, repeated:

1. Count every adjacent pair of symbols across all words, weighted by word frequency.
2. Find the single most frequent pair.
3. Merge every occurrence of that pair into one new symbol; add it to the vocabulary.
4. Repeat until the vocabulary reaches the target size (or no pair occurs more than once).

Each merge takes the two most-frequently-co-occurring smaller units and fuses them into
one bigger unit — starting from single characters and building up towards whole common
words, without ever being told what a "word" is in advance.

In [4]:
def get_pair_counts(freqs: dict[tuple[str, ...], int]) -> collections.Counter:
    pairs = collections.Counter()
    for symbols, freq in freqs.items():
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i + 1])] += freq
    return pairs


def merge_pair(pair, freqs: dict[tuple[str, ...], int]) -> dict[tuple[str, ...], int]:
    """Replace every adjacent occurrence of `pair` with the single merged symbol."""
    a, b = pair
    merged = a + b
    new_freqs: dict[tuple[str, ...], int] = {}
    for symbols, freq in freqs.items():
        new_symbols = []
        i = 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                new_symbols.append(merged)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        key = tuple(new_symbols)
        new_freqs[key] = new_freqs.get(key, 0) + freq
    return new_freqs


def train_bpe(corpus: str, target_vocab_size: int, verbose: bool = False):
    """Train BPE merges from scratch. Returns (merges, vocab) where `merges` is the
    ORDERED list of (pair, count) learned — order matters, encoding must apply the same
    merges in the same order they were learned."""
    freqs = word_freqs(corpus)
    vocab = set()
    for symbols in freqs:
        vocab.update(symbols)

    merges = []
    while len(vocab) < target_vocab_size:
        pair_counts = get_pair_counts(freqs)
        if not pair_counts:
            break
        best_pair, best_count = pair_counts.most_common(1)[0]
        if best_count < 2:
            break  # merging a pair that occurs once buys nothing
        freqs = merge_pair(best_pair, freqs)
        merged_token = best_pair[0] + best_pair[1]
        vocab.add(merged_token)
        merges.append((best_pair, best_count))
        if verbose:
            print(f"merge #{len(merges):2d}: {best_pair!r:22} -> {merged_token!r:14} "
                  f"(count={best_count:3d})  vocab_size={len(vocab)}")
    return merges, vocab


## 4. Actually run the merge loop

Target vocabulary size 60 (28 starting characters/symbols + 32 learned merges, capped at
60). This is the real, unmodified output of the training loop above — every merge listed
here is what the algorithm actually chose, not a hand-picked example.

In [5]:
merges, vocab = train_bpe(CORPUS, target_vocab_size=60, verbose=True)
print()
print("Final vocab size:", len(vocab))
print("Number of merges learned:", len(merges))


merge # 1: ('e', '</w>')          -> 'e</w>'        (count= 17)  vocab_size=28
merge # 2: ('t', 'h')             -> 'th'           (count= 15)  vocab_size=29
merge # 3: ('th', 'e</w>')        -> 'the</w>'      (count= 15)  vocab_size=30
merge # 4: ('s', '</w>')          -> 's</w>'        (count= 12)  vocab_size=31
merge # 5: ('r', '</w>')          -> 'r</w>'        (count= 12)  vocab_size=32
merge # 6: ('e', 'r</w>')         -> 'er</w>'       (count= 10)  vocab_size=33
merge # 7: ('o', 'w')             -> 'ow'           (count=  9)  vocab_size=34
merge # 8: ('t', 'o')             -> 'to'           (count=  9)  vocab_size=35
merge # 9: ('d', '</w>')          -> 'd</w>'        (count=  8)  vocab_size=36
merge #10: ('t', '</w>')          -> 't</w>'        (count=  7)  vocab_size=37
merge #11: ('e', 'n')             -> 'en'           (count=  7)  vocab_size=38
merge #12: ('l', 'ow')            -> 'low'          (count=  7)  vocab_size=39
merge #13: ('to', 'k')            -> 'tok'          

Read the trace: the very first merges are `('e', '</w>')` and `('t', 'h')` — `e` at
the end of a word and `t`+`h` are the single most frequent adjacent pairs in the corpus
(mostly from "the", which appears many times). By merge #3, `th` + `e</w>` fuse into the
whole word `the</w>` — the algorithm has "discovered" that "the" is a common enough whole
word to deserve its own single token, purely from co-occurrence counts, with no
dictionary or hand-coded notion of what a word is. Later merges build `low`, `token`,
`tokeniz`, `quick`, `fox`, `lazy` the same way — common whole words end up as single
tokens, exactly the behavior `notes.md` derives.

## 5. Encode / decode with the learned merges

To tokenize new text, split it into characters the same way training did, then apply the
learned merges **in the exact order they were learned** (a merge learned early takes
priority over one learned later, since it reflects a more corpus-wide-frequent pattern).
Decoding is the trivial inverse: concatenate the tokens and turn the end-of-word marker
back into a space.

In [6]:
def apply_merges(word: str, merges) -> list[str]:
    symbols = list(word.lower()) + [WORD_END]
    for (a, b), _count in merges:
        i = 0
        new_symbols = []
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                new_symbols.append(a + b)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        symbols = new_symbols
    return symbols


def encode(text: str, merges) -> list[str]:
    tokens = []
    for word in re.findall(r"\S+", text):
        tokens.extend(apply_merges(word, merges))
    return tokens


def decode(tokens: list[str]) -> str:
    text = "".join(tokens)
    text = text.replace(WORD_END, " ")
    return text.strip()


### Tokenize a brand-new sentence and verify the round trip

`"newest"`, `"lowers"`, and `"lowest"` never appeared verbatim in the training corpus
(only `"newer"`, `"lower"`, `"lowest"` did as separate words) — this checks that the
*learned subword pieces* recombine correctly on genuinely new input, not that the tokenizer
memorized whole training sentences.

In [7]:
test_sentence = "the newest tokenizer lowers the lowest words"
tokens = encode(test_sentence, merges)
print("Test sentence:", test_sentence)
print("Encoded tokens:", tokens)

decoded = decode(tokens)
print("Decoded:       ", repr(decoded))

assert decoded == test_sentence, f"ROUND TRIP FAILED: {decoded!r} != {test_sentence!r}"
print()
print("Round-trip check PASSED: decode(encode(text)) == text")


Test sentence: the newest tokenizer lowers the lowest words
Encoded tokens: ['the</w>', 'n', 'e', 'w', 'e', 'st</w>', 'tokeniz', 'er</w>', 'low', 'e', 'r', 's</w>', 'the</w>', 'low', 'e', 'st</w>', 'w', 'o', 'r', 'd', 's</w>']
Decoded:        'the newest tokenizer lowers the lowest words'

Round-trip check PASSED: decode(encode(text)) == text


## 6. Experiment: vocabulary size vs. average tokens-per-word

**Hypothesis (stated before running):** as target vocabulary size increases, more whole
words and larger subword chunks get merged into single tokens, so the average number of
tokens needed per word on held-out text should *decrease* — monotonically, until the
corpus is small enough that BPE runs out of any pair occurring more than once (at which
point adding more vocabulary slots buys nothing further).

**Setup:** train BPE from scratch on the same toy corpus at several target vocabulary
sizes, then tokenize one fixed held-out sentence (words drawn from the same domain but
not copied verbatim from the corpus) and measure `len(tokens) / len(words)`.

In [8]:
held_out = "the newer fox jumps over the tokenizer and watches the newest lazy model"
words = re.findall(r"\S+", held_out)

print(f"{'target_vocab':>12} {'actual_vocab':>13} {'merges':>7} {'tokens':>7} {'avg_tokens/word':>17}")
results = []
for target in [30, 45, 60, 80, 100, 130]:
    m, v = train_bpe(CORPUS, target_vocab_size=target)
    toks = encode(held_out, m)
    avg = len(toks) / len(words)
    results.append((target, len(v), len(m), len(toks), avg))
    print(f"{target:>12} {len(v):>13} {len(m):>7} {len(toks):>7} {avg:>17.3f}")


target_vocab  actual_vocab  merges  tokens   avg_tokens/word
          30            30       3      64             4.923
          45            45      18      47             3.615
          60            60      33      38             2.923
          80            80      53      31             2.385
         100            98      71      23             1.769
         130            98      71      23             1.769


**Result:** average tokens-per-word falls from **4.923** at `target_vocab=30` down to
**1.769** at `target_vocab=100`, then *stops decreasing* at `target_vocab=130` — the actual
vocabulary size plateaus at 98 (not 130) because the training loop's stopping rule
(`best_count < 2`, in `train_bpe` above) kicks in first: once every remaining adjacent pair
in this tiny corpus occurs only once, merging it would just be memorizing one specific
word, not learning a generalizable pattern, so training stops even though the requested
target (130) was never reached.

**Interpretation:** this confirms the hypothesis — bigger vocabularies do buy shorter
token sequences, by promoting more whole words and larger chunks to single tokens — but
also surfaces a real limit the derivation in `notes.md` predicts: a target vocabulary
size is a *ceiling*, not a *guarantee*; how large a vocabulary a corpus can actually
support depends on how much repeated structure (repeated adjacent pairs) that corpus
contains. A few KB of toy text simply cannot support the same vocabulary size a
multi-gigabyte real corpus can.

**Limitations:** one 13-word held-out sentence, one toy corpus, one random-free
deterministic run (BPE training here has no randomness, so this is not repeated across
seeds) — the trend (larger vocab -> fewer tokens/word, with a corpus-size-dependent
ceiling) is real and measured, but the specific numbers (4.923, 1.769, ...) are specific
to this tiny corpus and would look different, at a much larger vocabulary-size ceiling,
on real, large-scale training data.

## 7. Failure mode, demonstrated: wrong-domain text

The BPE merges above were learned entirely from English prose about foxes, dogs, and
tokenizers. Feeding it text from a different domain (Python source code) it never saw
patterns from shows the pathological result predicted in `notes.md`'s "Failure modes":
almost no merges apply, so the tokenizer falls back to near character-level splitting —
correct (nothing decodes incorrectly), but far less efficient than a tokenizer actually
trained on code would be.

In [9]:
merges_98, vocab_98 = train_bpe(CORPUS, target_vocab_size=98)  # the natural ceiling from the experiment above

wrong_domain_text = "def tokenize(x): return x.split()"
wrong_domain_tokens = encode(wrong_domain_text, merges_98)

print("Text:  ", wrong_domain_text)
print("Tokens:", wrong_domain_tokens)
print(f"{len(wrong_domain_tokens)} tokens for {len(wrong_domain_text.split())} whitespace-separated words "
      f"({len(wrong_domain_tokens) / len(wrong_domain_text.split()):.2f} tokens/word)")

# sanity: still round-trips correctly even though the split is pathological
assert decode(wrong_domain_tokens) == wrong_domain_text
print("(Round-trip still holds — the split is inefficient, not incorrect.)")


Text:   def tokenize(x): return x.split()
Tokens: ['d', 'e', 'f', '</w>', 'tokeniz', 'e', '(', 'x', ')', ':', '</w>', 'r', 'e', 't', 'u', 'r', 'n</w>', 'x', '.', 'spli', 't', '(', ')', '</w>']
24 tokens for 4 whitespace-separated words (6.00 tokens/word)
(Round-trip still holds — the split is inefficient, not incorrect.)


`24` tokens for `4` whitespace-separated "words" (6.0 tokens/word) — dramatically
worse than the `1.769` tokens/word this same vocabulary achieved on in-domain held-out
text above. `"tokenize"` itself partially survives (the learned `tokeniz` piece still
fires), but `"def"`, `"return"`, `"split"`, and every punctuation character fall back to
near single-character tokens, because the training corpus never saw Python keywords,
parentheses-as-syntax, or snake_case identifiers often enough (or at all) to learn merges
for them. The tokenizer still round-trips correctly — nothing decodes wrong — but the
sequence is far longer than it needs to be, exactly the tradeoff `notes.md`'s
"Failure modes" section describes.

## Summary

- A real BPE tokenizer was trained from scratch (plain Python, no library) on a small
  corpus, and its actual learned merges were inspected in order.
- A new sentence was tokenized with the learned vocabulary, and `decode(encode(text)) ==
  text` was verified as a genuine passing assertion — not asserted, checked.
- The vocabulary-size vs. tokens-per-word experiment was run for real, confirming the
  hypothesis and surfacing a real limitation (the corpus-size-dependent vocabulary
  ceiling) that only shows up by actually running the training loop.
- The wrong-domain failure mode was demonstrated concretely, not just described.

See `notes.md` for the full mathematical derivation, the contrast with `07-nlp/05-transformers-and-huggingface`'s
pretrained WordPiece tokenizer, and the remaining discussion (real-world usage, mental
model, questions to think about).